# 05 - Deterministic Matching

## Objective

Menguji aturan deterministic matching yang eksplisit pada dataset terstandardisasi. Fuzzy matching belum dilakukan pada tahap ini.

## Research Questions

1. Berapa banyak candidate pair yang dihasilkan setiap aturan exact matching?
2. Berapa banyak pair yang didukung oleh satu atau lebih aturan?
3. Apakah aturan exact menghasilkan konflik antar field?
4. Seberapa konsisten kandidat terhadap repeated `customer_id` sebagai audit internal?

## Rules

- R1: exact `email_std`; nilai tidak missing.
- R2: exact `phone_digits_std`; nilai tidak missing.
- R3: exact `name_key_std + dob_std`; kedua komponen tersedia.
- R4: exact `email_std + phone_digits_std`; kedua komponen tersedia.

R1 dan R2 dapat menghasilkan collision karena satu nilai dapat muncul pada beberapa baris. R3 dan R4 adalah composite rules yang diharapkan lebih selektif.

## Limitations

- Ground truth resmi belum tersedia.
- `customer_id` hanya dipakai untuk audit retrospektif, bukan untuk membuat match.
- Exact match tidak membuktikan entity yang sama secara absolut tanpa validasi eksternal.

In [1]:
from itertools import combinations
from pathlib import Path
import pandas as pd

DATA_CANDIDATES = [
    Path.cwd() / 'data' / 'processed' / 'crm_50000_customers_standardized.csv',
    Path.cwd().parent / 'data' / 'processed' / 'crm_50000_customers_standardized.csv',
]
DATA_PATH = next((path.resolve() for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Dataset terstandardisasi tidak ditemukan. Jalankan 04_standardization.ipynb terlebih dahulu.')

df = pd.read_csv(DATA_PATH).reset_index(names='row_index')
print('File:', DATA_PATH)
print('Shape:', df.shape)

File: C:\Users\User\Documents\Maganghub 2026\Bulan 1\Tes duplikasi\data\processed\crm_50000_customers_standardized.csv
Shape: (50000, 25)


## Experiment 1 - Eligibility of deterministic keys

Nilai missing tidak pernah dipakai sebagai bukti match. Composite key hanya valid jika seluruh komponennya tersedia.

In [3]:
name_dob_parts = df[['name_key_std', 'dob_std']]
df['name_dob_key'] = name_dob_parts.fillna('').astype('string').agg('|'.join, axis=1)
df.loc[name_dob_parts.isna().any(axis=1), 'name_dob_key'] = pd.NA

email_phone_parts = df[['email_std', 'phone_digits_std']]
df['email_phone_key'] = email_phone_parts.fillna('').astype('string').agg('|'.join, axis=1)
df.loc[email_phone_parts.isna().any(axis=1), 'email_phone_key'] = pd.NA

key_definitions = {
    'email_std': 'R1_exact_email',
    'phone_digits_std': 'R2_exact_phone',
    'name_dob_key': 'R3_name_plus_dob',
    'email_phone_key': 'R4_email_plus_phone',
}
eligibility_rows = []
for key, rule in key_definitions.items():
    values = df[key].dropna()
    counts = values.value_counts()
    repeated = counts[counts > 1]
    eligibility_rows.append({
        'rule': rule,
        'key': key,
        'eligible_rows': int(values.size),
        'unique_key_values': int(values.nunique()),
        'repeated_key_values': int(len(repeated)),
        'candidate_pairs': int((repeated * (repeated - 1) // 2).sum()),
        'largest_block': int(counts.max()) if len(counts) else 0,
    })
eligibility_summary = pd.DataFrame(eligibility_rows)
eligibility_summary

,rule,key,eligible_rows,unique_key_values,repeated_key_values,candidate_pairs,largest_block
0,R1_exact_email,email_std,48960,46363,2102,3422,10
1,R2_exact_phone,phone_digits_std,50000,46777,2161,5509,9
2,R3_name_plus_dob,name_dob_key,50000,48565,1388,1482,3
3,R4_email_plus_phone,email_phone_key,48960,47200,1695,1826,4


## Experiment 2 - Candidate pair generation

Pair dibuat di dalam repeated block R1, R2, R3, dan R4. Row index adalah referensi teknis dan tidak membuka data customer.

Output berisi satu baris per pair dan aturan yang mendukung pair tersebut.

In [4]:
pair_rules = {}
for key, rule in [
    ('email_std', 'R1_exact_email'),
    ('phone_digits_std', 'R2_exact_phone'),
    ('name_dob_key', 'R3_name_plus_dob'),
    ('email_phone_key', 'R4_email_plus_phone'),
]:
    for _, block in df.dropna(subset=[key]).groupby(key, sort=False):
        row_indices = sorted(block['row_index'].tolist())
        for left_index, right_index in combinations(row_indices, 2):
            pair_rules.setdefault((left_index, right_index), set()).add(rule)

pair_rows = []
for (left_index, right_index), rules in sorted(pair_rules.items()):
    pair_rows.append({
        'left_row_index': left_index,
        'right_row_index': right_index,
        'supporting_rules': '|'.join(sorted(rules)),
        'rule_count': len(rules),
    })
candidate_pairs = pd.DataFrame(pair_rows)
if candidate_pairs.empty:
    candidate_pairs = pd.DataFrame(columns=['left_row_index', 'right_row_index', 'supporting_rules', 'rule_count'])

print('Candidate pairs:', len(candidate_pairs))
candidate_pairs['rule_count'].value_counts().sort_index().rename('pair_count').to_frame()

Candidate pairs: 7106


,pair_count
rule_count,
1,5245
2,35
3,380
4,1446


In [5]:
pair_support_summary = (
    candidate_pairs.groupby('supporting_rules', dropna=False)
    .size()
    .reset_index(name='candidate_pair_count')
    .sort_values('candidate_pair_count', ascending=False)
)
pair_support_summary

,supporting_rules,candidate_pair_count
3,R2_exact_phone,3648
0,R1_exact_email,1596
1,R1_exact_email|R2_exact_phone|R3_name_plus_dob...,1446
2,R1_exact_email|R2_exact_phone|R4_email_plus_phone,380
4,R2_exact_phone|R3_name_plus_dob,35
5,R3_name_plus_dob,1


## Experiment 3 - Internal audit against customer_id

Audit ini bukan ground truth resmi. Label hanya menunjukkan apakah dua row candidate memiliki `customer_id` yang sama. Label tidak dipakai untuk membuat match.

In [6]:
if candidate_pairs.empty:
    audit_pairs = candidate_pairs.copy()
    audit_summary = pd.DataFrame(columns=['audit_label', 'pair_count', 'percentage'])
else:
    audit_pairs = candidate_pairs.merge(df[['row_index', 'customer_id']], left_on='left_row_index', right_on='row_index', how='left')
    audit_pairs = audit_pairs.rename(columns={'customer_id': 'left_customer_id'}).drop(columns='row_index')
    audit_pairs = audit_pairs.merge(df[['row_index', 'customer_id']], left_on='right_row_index', right_on='row_index', how='left')
    audit_pairs = audit_pairs.rename(columns={'customer_id': 'right_customer_id'}).drop(columns='row_index')
    audit_pairs['same_customer_id'] = audit_pairs['left_customer_id'] == audit_pairs['right_customer_id']
    audit_summary = (
        audit_pairs['same_customer_id'].map({True: 'same_customer_id', False: 'different_customer_id'})
        .value_counts()
        .rename_axis('audit_label')
        .reset_index(name='pair_count')
    )
    audit_summary['percentage'] = audit_summary['pair_count'].div(len(audit_pairs)).mul(100)
audit_summary.round(2)

,audit_label,pair_count,percentage
0,different_customer_id,5239,73.73
1,same_customer_id,1867,26.27


In [7]:
if candidate_pairs.empty:
    conflict_summary = pd.DataFrame(columns=['supporting_rules', 'pair_count', 'different_customer_id_count'])
else:
    conflict_summary = (
        audit_pairs.assign(different_customer_id=~audit_pairs['same_customer_id'])
        .groupby('supporting_rules')
        .agg(pair_count=('same_customer_id', 'size'), different_customer_id_count=('different_customer_id', 'sum'))
        .reset_index()
    )
conflict_summary

,supporting_rules,pair_count,different_customer_id_count
0,R1_exact_email,1596,1596
1,R1_exact_email|R2_exact_phone|R3_name_plus_dob...,1446,0
2,R1_exact_email|R2_exact_phone|R4_email_plus_phone,380,0
3,R2_exact_phone,3648,3642
4,R2_exact_phone|R3_name_plus_dob,35,0
5,R3_name_plus_dob,1,1


# Result, Analysis, and Decision

## Result aktual

- R1 exact email: `3.422` candidate pair dari `2.102` repeated key values.
- R2 exact phone: `5.509` candidate pair dari `2.161` repeated key values.
- R3 name + DOB: `1.482` candidate pair dari `1.388` repeated key values.
- R4 email + phone: `1.826` candidate pair dari `1.695` repeated key values.
- Union candidate pair setelah deduplication: `7.106` pair.
- Dukungan satu rule: `5.245` pair; dua rule: `35` pair; tiga rule: `380` pair; empat rule: `1.446` pair.
- Audit internal: `1.867` pair atau `26,27%` memiliki `customer_id` sama; `5.239` pair atau `73,73%` memiliki `customer_id` berbeda.

## Analysis

- Exact email dan exact phone saja menghasilkan banyak collision pada snapshot ini. Keduanya tidak layak dijadikan automatic match rule tanpa bukti tambahan.
- Pair yang didukung kombinasi lebih kuat, terutama R4 atau beberapa rule sekaligus, konsisten dengan `customer_id` pada audit internal.
- Audit internal bukan ground truth resmi. Perbedaan `customer_id` dapat berarti false positive, tetapi juga dapat mencerminkan bahwa `customer_id` bukan master identifier yang valid.

## Decision

Belum ada master record yang dibuat dan belum ada baris yang dihapus. Untuk eksperimen berikutnya, gunakan aturan composite atau multi-rule sebagai kandidat prioritas; jangan otomatis menerima pair yang hanya didukung exact email atau exact phone.

Ground truth belum tersedia, sehingga angka audit tidak boleh disebut precision atau recall resmi. Jumlah match terbesar juga bukan dasar tunggal memilih aturan.

## Next Experiment

Jika candidate pair deterministic yang lebih ketat masih memiliki coverage rendah atau konflik, lakukan `06_fuzzy_matching.ipynb` hanya pada candidate pair yang sudah diblok. Jika coverage deterministic sudah cukup, fuzzy matching dapat ditunda.